In [1]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import os
import tensorflow as tf

2026-03-11 07:58:35.249250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773241115.265086 1918495 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773241115.270183 1918495 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773241115.284466 1918495 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773241115.284482 1918495 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773241115.284484 1918495 computation_placer.cc:177] computation placer alr

In [2]:
#Transform density
def transform(density,density_ref,transform_type):
    dens = density
    dref  = density_ref
    ttype = transform_type
    if ttype=='value':
        dtrans = density
    elif ttype=='sqrt':
        dtrans = np.sqrt(np.abs(density))
    elif ttype=='log':
        dtrans  = np.log(np.abs(density))
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        dtrans = (dens-dval)/dsqrt
    else:
        raise RuntimeError('bad transform type')
    return dtrans


def inverse_transform(density_trans,density_ref,transform_type):
    dref   = density_ref
    ttype  = transform_type
    dtrans = density_trans
    if ttype=='value':
        density = dtrans
    elif ttype=='sqrt':
        density = dtrans**2
    elif ttype=='log':
        density = np.log(np.abs(dtrans))
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        density = dval + dsqrt*dtrans
    else:
        raise RuntimeError('bad transform type')
    return density



In [3]:
####################################################################################################################################################
def stochastic_density(d,N):
    # poisson model
    #  accurate and fast for all values of N
    # N  = number of MC samples
    assert isinstance(d,np.ndarray)
    assert isinstance(N,(int,float,np.int64,np.float64))
    assert N>0
    ds = np.random.poisson(N*d)/N
    ds*= d.sum()/ds.sum()
    return ds
#end def stochastic_density

####################################################################################################################################################

In [4]:
import numpy as np
import scipy.ndimage
import h5py
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================
SHAPE = (116, 116, 72)  # Your specific grid dimensions
N_ELECTRONS = 50.0    # Target electron count
NUM_TRAIN = 1000
NUM_VAL = 200
MC_SAMPLES = 10**6    # N parameter for stochastic_density



# ==========================================
# 3. SYNTHETIC GENERATION LOGIC
# ==========================================

def create_fake_dft(shape):
    print(f"  > Generating Synthetic DFT Reference {shape}...")
    grid = np.zeros(shape, dtype=np.float32)
    
    centers = [
        (shape[0]//2, shape[1]//2, shape[2]//2),       
        (shape[0]//3, shape[1]//2, shape[2]//2),       
        (shape[0]//2, shape[1]//2 + 15, shape[2]//2)   
    ]
    
    z, y, x = np.indices(shape)
    
    for (z0, y0, x0) in centers:
        r2 = (z - z0)**2 + (y - y0)**2 + (x - x0)**2
        grid += np.exp(-r2 / (2 * 6.0**2)) 
        
    # Apply floor FIRST
    grid = np.maximum(grid, 1e-12)
    
    # Normalize SECOND, strictly using float64
    grid *= (N_ELECTRONS / np.sum(grid, dtype=np.float64))
    
    return grid.astype(np.float32)

def generate_blobs(shape, scale=6):
    """
    Creates coherent, low-frequency fluctuations to simulate 
    Physical Electron Correlation (the signal we want to learn).
    """
    # 1. Low Res Grid
    low_res_shape = np.maximum(np.array(shape) // scale, 1)
    field = np.random.uniform(-1.0, 1.0, size=low_res_shape)
    
    # 2. Upsample to smooth it out
    zoom = np.array(shape) / np.array(low_res_shape)
    blobs = scipy.ndimage.zoom(field, zoom, order=2)
    
    # 3. Crop to exact size
    return blobs[:shape[0], :shape[1], :shape[2]]

def generate_dataset(n_items, dft_ref, n_samples):
    """
    Generates (Input, Target) pairs in RESIDUAL space.
    """
    x_data = np.zeros((n_items, *dft_ref.shape), dtype=np.float32)
    y_data = np.zeros((n_items, *dft_ref.shape), dtype=np.float32)
    
    print(f"  > Generating {n_items} pairs using N={n_samples}...")
    
    for i in range(n_items):
        # A. Create "True VMC" Density (DFT + Signal Blobs)
        # -------------------------------------------------
        # We perturb the DFT. This "True" density is what we want the model 
        # to predict, NOT the DFT itself.
        blobs = generate_blobs(dft_ref.shape)
        perturbation_strength = np.random.uniform(0.10, 0.20)
        
        # True = DFT * (1 + blobs)
        true_rho = dft_ref * (1.0 + (perturbation_strength * blobs))
        true_rho = np.maximum(true_rho, 1e-12) # Physics constraint
        
        # Renormalize (Charge Conservation)
        true_rho *= (N_ELECTRONS / np.sum(true_rho))
        
        # B. Create Noisy Input (Using YOUR stochastic_density)
        # ----------------------------------------------------
        # Simulate the VMC sampling process
        noisy_rho = stochastic_density(true_rho, n_samples)
        
        # C. Transform to Residual Space (Using YOUR transform)
        # ---------------------------------------------------
        # Input: The noisy, sampled data transformed
        x_res = transform(noisy_rho, dft_ref, 'residual_noise')
        
        # Target: The CLEAN, perturbed data transformed.
        # This teaches the model: "Given this noisy residual, find the 
        # underlying smooth residual deviation from DFT."
        y_res = transform(true_rho, dft_ref, 'residual_noise')
        
        x_data[i] = x_res
        y_data[i] = y_res
        
        if i % 100 == 0 and i > 0:
            print(f"    ...batch {i} complete")
            
    return x_data, y_data

# ==========================================
# 4. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    print("--- STARTING SYNTHETIC DATA GENERATION ---")
    
    # 1. Create the Mock DFT Reference
    dft_ref = create_fake_dft(SHAPE)
    
    # 2. Generate Training Data
    print("\n[Training Data]")
    x_train, y_train = generate_dataset(NUM_TRAIN, dft_ref, MC_SAMPLES)
    
    # 3. Generate Validation Data
    print("\n[Validation Data]")
    x_val, y_val = generate_dataset(NUM_VAL, dft_ref, MC_SAMPLES)
    
    # 4. Save to Disk
    filename = f"NO_DFT/Synthetic_Transformed_N{MC_SAMPLES}_MD.npz"
    print(f"\n>>> Saving {filename}...")
    
    np.savez(filename,
             x_train=x_train,
             y_train=y_train,
             x_val=x_val,
             y_val=y_val,
             dft_ref=dft_ref) # Save ref for inverse_transform later
             
    print("Done.")

--- STARTING SYNTHETIC DATA GENERATION ---
  > Generating Synthetic DFT Reference (116, 116, 72)...

[Training Data]
  > Generating 1000 pairs using N=1000000...
    ...batch 100 complete
    ...batch 200 complete
    ...batch 300 complete
    ...batch 400 complete
    ...batch 500 complete
    ...batch 600 complete
    ...batch 700 complete
    ...batch 800 complete
    ...batch 900 complete

[Validation Data]
  > Generating 200 pairs using N=1000000...
    ...batch 100 complete

>>> Saving /NO_DFT/Synthetic_Transformed_N1000000_MD.npz...


FileNotFoundError: [Errno 2] No such file or directory: '/NO_DFT/Synthetic_Transformed_N1000000_MD.npz'

In [6]:
    # 4. Save to Disk
    filename = f"NO_DFT/Synthetic_Transformed_N{MC_SAMPLES}_MD.npz"
    print(f"\n>>> Saving {filename}...")
    
    np.savez(filename,
             x_train=x_train,
             y_train=y_train,
             x_val=x_val,
             y_val=y_val,
             dft_ref=dft_ref) # Save ref for inverse_transform later
             
    print("Done.")


>>> Saving NO_DFT/Synthetic_Transformed_N1000000_MD.npz...
Done.
